CURRENT LIMITATION: We still have not solved for a case like Harry Potter where archetype data has one instance and IMDB has many.

Builds and exports `df_model`: a story-level dataframe where each row is one story and columns are the proportions of each base archetype across its cast.

**Pipeline:**
1. Load character archetype data from `archetypometricsdata2000.mat`
2. Load IMDB metadata — filtered to Comedy/Drama entries only before joining
3. Join stories → IMDB titles (exact match, fuzzy fallback)
4. Keep only stories that are *exclusively* Comedy or *exclusively* Drama (not both)
5. Collapse 232 compound archetypes to 6 base types; normalize to per-story proportions
6. Export `df_model.csv`

In [4]:
import scipy.io
import pandas as pd
from difflib import get_close_matches

Each of the 2,000 characters has a `primary_archetype` label and belongs to one of 341 stories. The `character_story_typenames` field encodes both as `"CharName/StoryName"` — we split on `/` to isolate the story name.

In [5]:
mat = scipy.io.loadmat('../datasets/archetypometricsdata2000.mat', simplify_cells=True)
chars_raw       = mat['data_characters']
archetype_space = mat['data_archetype_space']

char_story_pairs = list(chars_raw['character_story_typenames'])

df_chars = pd.DataFrame({
    'character_name':    list(chars_raw['character_typenames']),
    'story_name':        [p.split('/', 1)[1] if '/' in p else p for p in char_story_pairs],
    'primary_archetype': list(archetype_space['character_archetypes_ordered']),
})

print(f"Characters: {len(df_chars):,}  |  Unique stories: {df_chars['story_name'].nunique()}")

Characters: 2,000  |  Unique stories: 341


**Key fix:** we restrict the IMDB lookup to rows that already have Comedy or Drama in their genres *before* building the title index. Without this, a story like *The Office* could match an earlier IMDB entry with the same name (e.g. a documentary) that has no genre tag — and would be silently mislabeled or dropped.

In [6]:
df_imdb = pd.read_csv('../datasets/title.basics.tsv', sep='\t', na_values='\\N', low_memory=False)
df_imdb['startYear'] = pd.to_numeric(df_imdb['startYear'], errors='coerce')

# Filter 1: title types that could appear in archetypometrics
relevant_types = {'movie', 'tvSeries', 'tvMiniSeries', 'tvMovie', 'tvSpecial', 'short', 'tvShort'}
df_imdb = df_imdb[df_imdb['titleType'].isin(relevant_types)].copy()

# Filter 2: must have Comedy or Drama in genres — ensures we always match to
# the right entry when a title name appears multiple times in IMDB
genres = df_imdb['genres'].fillna('')
df_imdb = df_imdb[genres.str.contains('Comedy') | genres.str.contains('Drama')].copy()

# Build lookup: lowercase title → earliest matching IMDB row
df_imdb['title_lower'] = df_imdb['primaryTitle'].str.lower().str.strip()
df_imdb = df_imdb.sort_values('startYear', na_position='last')
imdb_lookup = df_imdb.drop_duplicates(subset='title_lower', keep='first').set_index('title_lower')

print(f"IMDB Comedy/Drama entries: {len(imdb_lookup):,}")

IMDB Comedy/Drama entries: 832,140


**Pass 1** — exact case-insensitive match  
**Pass 2** — fuzzy fallback via `difflib.get_close_matches` (cutoff = 0.85)  

Characters from unmatched stories will have `NaN` IMDB columns and are dropped when we attach the label later.

In [7]:
unique_stories = df_chars['story_name'].unique()
story_lower    = {s: s.lower().strip() for s in unique_stories}

# Pass 1: exact match
exact_map = {s: imdb_lookup.loc[lower]
             for s, lower in story_lower.items()
             if lower in imdb_lookup.index}

# Pass 2: fuzzy match for remaining stories
all_imdb_titles_lower = [t for t in imdb_lookup.index if isinstance(t, str)]
fuzzy_map = {}
for story in (s for s in unique_stories if s not in exact_map):
    matches = get_close_matches(story_lower[story], all_imdb_titles_lower, n=1, cutoff=0.85)
    if matches:
        fuzzy_map[story] = imdb_lookup.loc[matches[0]]

print(f"Exact: {len(exact_map)}  |  Fuzzy: {len(fuzzy_map)}  |  "
      f"Total matched: {len(exact_map)+len(fuzzy_map)} / {len(unique_stories)}")
print(f"Unmatched: {[s for s in unique_stories if s not in exact_map and s not in fuzzy_map]}")

# Build story-level IMDB table and left-join to characters
all_matches  = {**exact_map, **fuzzy_map}
match_source = {**{s: 'exact' for s in exact_map}, **{s: 'fuzzy' for s in fuzzy_map}}
imdb_cols    = ['tconst', 'titleType', 'primaryTitle', 'startYear', 'genres']

df_story_imdb = pd.DataFrame([
    {'story_name': s, 'match_type': match_source[s],
     **{f'imdb_{c}': row[c] for c in imdb_cols}}
    for s, row in all_matches.items()
])

df_joined = df_chars.merge(df_story_imdb, on='story_name', how='left')
print(f"\nMatch rate: {df_joined['imdb_tconst'].notna().mean():.1%} "
      f"({df_joined['imdb_tconst'].notna().sum():,} / {len(df_joined):,} characters)")

Exact: 307  |  Fuzzy: 20  |  Total matched: 327 / 341
Unmatched: ['Star Wars: The Last Jedi', 'Jurassic World', 'House, M.D.', 'Law & Order: SVU', 'Harry Potter', 'Agents of S.H.I.E.L.D.', 'The Boondock Saints', 'Firefly + Serenity', 'Fullmetal Alchemist: Brotherhood', 'Sailor Moon', 'Calvin and Hobbes', 'Mad Max: Fury Road', 'Nineteen Eighty-Four', 'Atomic Blonde']

Match rate: 95.2% (1,905 / 2,000 characters)


In [11]:
df_joined.head()
df_joined.to_csv('../datasets/characters_with_imdb.csv', index=False)

- **Drama XOR Comedy** — stories tagged as both are ambiguous labels; exclude them
- **Base archetypes** — compound labels like `Angel-Hero` are collapsed to their dominant base type (first match in the priority list)
- **Top 6** — rare base types are lumped into `Other` and then dropped; keeping only the 6 most common reduces noise
- **Proportions** — raw counts are divided by cast size so stories with different numbers of characters are comparable

In [8]:
# Keep only Drama XOR Comedy stories
genres_col = df_joined['imdb_genres'].fillna('')
is_drama   = genres_col.str.contains('Drama')
is_comedy  = genres_col.str.contains('Comedy')
df_filtered = df_joined[is_drama ^ is_comedy].copy()

# Collapse compound archetype labels to a single base type
base_archetypes = ['Hero', 'Demon', 'Fool', 'Angel', 'Adventurer',
                   'Traditionalist', 'Diva', 'Outcast', 'Brute', 'Lone Wolf', 'Geek']

def extract_base(label):
    for base in base_archetypes:
        if base in label:
            return base
    return 'Other'

df_filtered['base_archetype'] = df_filtered['primary_archetype'].apply(extract_base)

# Keep the top 6 most frequent; everything else → Other (then dropped)
top_6 = df_filtered['base_archetype'].value_counts().head(6).index.tolist()
print(f"Top 6 base archetypes: {top_6}")
df_filtered['base_archetype'] = df_filtered['base_archetype'].apply(
    lambda x: x if x in top_6 else 'Other'
)

# Story-level archetype proportions
counts = (df_filtered
    .groupby(['story_name', 'base_archetype'])
    .size()
    .unstack(fill_value=0))

df_archetype_props = counts.div(counts.sum(axis=1), axis=0)
df_archetype_props = df_archetype_props.drop(columns=['Other'], errors='ignore')

# Attach is_comedy label (1 = Comedy, 0 = Drama)
df_archetype_props['is_comedy'] = (
    df_filtered.groupby('story_name')['imdb_genres']
    .first()
    .str.contains('Comedy')
    .astype(int)
)

df_model = df_archetype_props.dropna(subset=['is_comedy'])

print(f"\ndf_model: {df_model.shape[0]} stories × {df_model.shape[1]-1} archetype features")
print(df_model['is_comedy'].value_counts().rename({0: 'Drama', 1: 'Comedy'}).to_string())
df_model.head()

Top 6 base archetypes: ['Hero', 'Demon', 'Angel', 'Adventurer', 'Fool', 'Traditionalist']

df_model: 283 stories × 6 archetype features
is_comedy
Drama     188
Comedy     95


base_archetype,Adventurer,Angel,Demon,Fool,Hero,Traditionalist,is_comedy
story_name,,,,,,,
30 Rock,0.0,0.166667,0.333333,0.166667,0.333333,0.0,1
8 Mile,0.0,0.000000,0.000000,0.000000,1.000000,0.0,0
A Nightmare on Elm Street,0.2,0.000000,0.200000,0.000000,0.600000,0.0,0
A Series of Unfortunate Events,0.0,0.000000,0.333333,0.000000,0.666667,0.0,1
A Star Is Born,0.0,0.000000,0.500000,0.000000,0.500000,0.0,0


The story name is the index — set `index=True` so it round-trips cleanly when loaded in `model.ipynb`.

In [9]:
df_model.to_csv('df_model.csv', index=True)
print(f"Exported df_model.csv — {df_model.shape[0]} rows, {df_model.shape[1]} columns")

Exported df_model.csv — 283 rows, 7 columns
